In [ ]:
# imports

import os
from dotenv import load_dotenv
#from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]


# Short Summary of a Paragraph

Analysis of a short paragraph to provide a short summary.

Also guessing from which famous book is the paragraph taken.

In [ ]:
# Step 1: Create your prompts

system_prompt = """
You are a helpful assistant that analyzes the contents of a paragraph
and provides:

1. A short summary
2. The likely source book if known

Format the response clearly in markdown using headings and bullet points.

If you don't know the source book, say you don't know.
Do not wrap the markdown in a code block.
"""

user_prompt = """
Here are the contents of the paragraph.
R. BENNET was among the earliest of those who waited on Mr. Bingley. He had always intended to visit him, 
though to the last always assuring his wife that he should not go; and till the evening after the visit was 
paid she had no knowledge of it. It was then disclosed in the following manner. 
Observing his second daughter employed in trimming a hat, he suddenly addressed her with,—
“I hope Mr. Bingley will like it, Lizzy.”
“We are not in a way to know what Mr. Bingley likes,” said her mother, resentfully, “since we are not to visit.”
“But you forget, mamma,” said Elizabeth, “that we shall meet him at the assemblies, and that 
Mrs.Long has promised to introduce him.”
“I do not believe Mrs. Long will do any such thing. She has two nieces of her own. She is a selfish, hypocritical
 woman, and I have no opinion of her.”
“No more have I,” said Mr. Bennet; “and I am glad to find that you do not depend on her serving you.”
Mrs. Bennet deigned not to make any reply; but, unable to contain herself, began scolding one of her daughters.
“Don’t keep coughing so, Kitty, for heaven’s sake! Have a little compassion on my nerves. You tear them to pieces.”
“Kitty has no discretion in her coughs,” said her father; “she times them ill.”
“I do not cough for my own amusement,” replied Kitty, fretfully. “When is your next ball to be, Lizzy?”
“To-morrow fortnight.”
“Ay, so it is,” cried her mother, “and Mrs. Long does not come back till the day before; 
so, it will be impossible for her to introduce him, for she will not know him herself.”
“Then, my dear, you may have the advantage of your friend, and introduce Mr. Bingley to her.”
“Impossible, Mr. Bennet, impossible, when I am not acquainted with him myself; how can you be so teasing?”
“I honour your circumspection. A fortnight’s acquaintance is certainly very little. 
One cannot know what a man really is by the end of a fortnight. 
But if we do not venture, somebody else will; and after all, Mrs. Long and her nieces must stand their chance; 
and, therefore,as she will think it an act of kindness, if you decline the office, I will take it on myself.”    
"""

# Step 2: Make the messages list

messages = [{"role":"system","content":system_prompt},{"role":"user","content":user_prompt}] # fill this in

# Step 3: Call OpenAI
openai = OpenAI()
response = openai.chat.completions.create(model = "gpt-4.1-nano", messages = messages)

# Step 4: print the result
display(Markdown(response.choices[0].message.content))